# Context-aware trajectory compression (CADP)

Douglas–Peucker keeps the points that carry **spatial** information. A point on
a straight leg is spatially redundant, yet it may be the moment the waves rose
or the water shoaled, and once dropped that information is gone from every
analysis downstream. **CADP** runs Douglas–Peucker and then adds back every
point whose removal would misrepresent a context variable by more than its
threshold.

This notebook walks through it on real data, AIS vessel positions with
WAVEWATCH III sea state for the English Channel and Bay of Biscay:

1. **Data**: where to download it, and the licence each part carries
2. **Annotation**: attaching bottom depth, wave height and wave direction to every position
3. **Compress**: one call, two modes
4. **One vessel up close**: which points are kept, and why
5. **Choosing thresholds**: what each threshold buys
6. **With MovingPandas**: `Trajectory` and `TrajectoryCollection` in, the same type out
7. **Speed**

`catraj` builds on [MovingPandas](https://movingpandas.org):

> Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)

The method is from:

> Mehri, S., Hooshangi, N., Mahdizadeh Gharakhanlou, N. (2025). *A Novel Context-Aware Douglas–Peucker (CADP) Trajectory Compression Method.* ISPRS Int. J. Geo-Inf. 14(2), 58. [doi:10.3390/ijgi14020058](https://doi.org/10.3390/ijgi14020058)

The annotation step is covered in depth in [`01_trajectory_annotation.ipynb`](01_trajectory_annotation.ipynb).

---
## 1 · Data

Everything comes from one public archive:

> Ray, C., Dréo, R., Camossi, E., Jousselme, A.-L., Iphar, C. (2019). *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and Reconnaissance.* Data in Brief 25, 104141. [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141)
> Data: Zenodo, [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595), record <https://zenodo.org/records/1167595>

It covers 1 October 2015 to 31 March 2016, longitude −10° to 0°, latitude 45° to 51°. This notebook needs two of its files:

| Zenodo file | Size | What is used | Source | Licence |
|---|---|---|---|---|
| [`[P1] AIS Data.zip`](https://zenodo.org/records/1167595/files/%5BP1%5D%20AIS%20Data.zip?download=1) | 201 MB (1.1 GB unzipped) | `nari_dynamic.csv`: AIS positions, 19 million messages | Naval Academy Research Institute (NARI), Brest, France | **[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)** |
| [`[E1] Ocean Conditions.zip`](https://zenodo.org/records/1167595/files/%5BE1%5D%20Ocean%20Conditions.zip?download=1) | 570 MB (4.2 GB unzipped) | `oc_february.csv`: sea state, 3-hourly, 43,206 sea points | SHOM – IFREMER, WAVEWATCH III hindcast | Public, **citation mandatory** (below) |

**What the licences require:**

* **AIS, CC BY-NC-SA 4.0.** *BY*: attribute the Naval Academy Research Institute (NARI) and cite the dataset above. *NC*: **no commercial use**. *SA*: anything derived from it, including annotated or compressed trajectories and figures made from them, must be shared under the same licence. The BSD licence of `catraj`'s code does not extend to this data.
* **Ocean conditions**: public, but every use must cite
  Boudière, E., Maisondieu, C., Ardhuin, F., Accensi, M., Pineau-Guillou, L., Lepesqueur, J. (2013). *A suitable metocean hindcast database for the design of Marine energy converters.* International Journal of Marine Energy 3–4, e40–e52. [doi:10.1016/j.ijome.2013.11.010](https://doi.org/10.1016/j.ijome.2013.11.010)

The data is **not** part of this repository. It goes in `data/zenodo_1167595/`, which git ignores.

**Getting it.** Any one of:

* from a terminal, at the repository root: `python scripts/download_data.py cadp` (checksums verified, resumes if interrupted);
* set `DOWNLOAD = True` in the next cell, which fetches both archives (about 770 MB) and extracts only what is needed;
* download them by hand and unzip.

Every step, disk space, checksums and troubleshooting are in [**getting_the_data.md**](../docs/getting_the_data.md). Either way, these two files must exist:

```
data/zenodo_1167595/P1_AIS_Data/nari_dynamic.csv
data/zenodo_1167595/E1_ocean/oc_february.csv
```

**Then prepare them.** Run `python scripts/prepare_zenodo.py` once (it needs `pyarrow`). It renames the archive's coded columns to what they measure, sets the "not available" codes to missing values and median-filters the columns used here, and writes the two files this notebook reads:

```
data/zenodo_1167595/clean/ais_dynamic.parquet
data/zenodo_1167595/clean/ocean_february.parquet
```

In [1]:
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")   # see catraj/__init__.py

import hashlib, time, urllib.request, zipfile
from pathlib import Path

import geopandas as gpd
import holoviews as hv
import hvplot.pandas  # noqa: F401  -- registers .hvplot
import numpy as np
import pandas as pd
from pyproj import Transformer

from catraj.annotation import DLT, IDW, annotate_points
from catraj.compression import (Context, cadp_select, compress, compression_ratio,
                                context_errors, douglas_peucker, dtw)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "data" / "zenodo_1167595"
# Cleaned files written by scripts/prepare_zenodo.py: descriptive column names,
# "not available" codes as NaN, noise filtered
AIS_PARQUET = DATA / "clean" / "ais_dynamic.parquet"
OCEAN_PARQUET = DATA / "clean" / "ocean_february.parquet"

# ---- settings ---------------------------------------------------------------
DOWNLOAD = False                    # True: fetch the two archives from Zenodo (~770 MB)
START, END = "2016-02-01", "2016-02-04"   # the time window to work on (UTC); the
                                          # ocean file covers February 2016 only
CRS = 32630                         # UTM zone 30N: metres, as distance thresholds need
hv.extension("bokeh")               # interactive plots: zoom, pan, hover, click the legend
ORANGE, GREEN, INK = "#e16f24", "#1a7f37", "#0d1117"

In [2]:
ZENODO = "https://zenodo.org/records/1167595/files/"
ARCHIVES = {   # Zenodo name -> (md5 from the record, members to extract, target folder)
    "[P1] AIS Data.zip": ("90bec666a3c3474e18bb57676348b655",
                          ["nari_dynamic.csv", "README_parameters.txt"], DATA / "P1_AIS_Data"),
    "[E1] Ocean Conditions.zip": ("20a1c8f8a6bbbd9bdf9821f938f22add",
                                  ["oc_february.csv", "README_parameters.txt"], DATA / "E1_ocean"),
}


def fetch(name, md5, members, target):
    zpath = DATA / name.replace("[", "").replace("] ", "_").replace(" ", "_")
    if not zpath.exists():
        url = ZENODO + urllib.request.quote(name) + "?download=1"
        print(f"downloading {name} ...")
        with urllib.request.urlopen(url) as r, open(zpath, "wb") as f:
            size, done = int(r.headers.get("Content-Length", 0)), 0
            while chunk := r.read(8 << 20):
                f.write(chunk)
                done += len(chunk)
                print(f"\r  {done / 1e6:,.0f} / {size / 1e6:,.0f} MB", end="")
        print()
    digest = hashlib.md5()
    with open(zpath, "rb") as f:
        while chunk := f.read(8 << 20):
            digest.update(chunk)
    if digest.hexdigest() != md5:
        raise RuntimeError(f"{zpath.name}: checksum mismatch, delete it and download again")
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in members:
            if not (target / m).exists():
                z.extract(m, target)
    print(f"{name}: checksum ok, extracted {members} to {target.relative_to(ROOT)}")


if DOWNLOAD:
    DATA.mkdir(parents=True, exist_ok=True)
    for name, spec in ARCHIVES.items():
        fetch(name, *spec)

missing = [p for p in (AIS_PARQUET, OCEAN_PARQUET) if not p.exists()]
if missing:
    raise FileNotFoundError("missing: " + ", ".join(str(p.relative_to(ROOT)) for p in missing)
                            + "\nset DOWNLOAD = True above (or run  python scripts/download_data.py cadp),"
                            " then run  python scripts/prepare_zenodo.py"
                            "\n(see docs/getting_the_data.md)")
print("data present:", AIS_PARQUET.relative_to(ROOT), "and", OCEAN_PARQUET.relative_to(ROOT))

data present: data\zenodo_1167595\P1_AIS_Data\nari_dynamic.csv and data\zenodo_1167595\E1_ocean\oc_february.csv


### AIS positions

The window is cut out, exact duplicates (same vessel, same timestamp) are
dropped, AIS "not available" codes become missing values (speed 102.3 kn,
heading 511), and positions are projected to UTM 30N. Douglas–Peucker's
threshold is a distance in metres, so this matters: in degrees, a step east is
about 0.7 of a step north at 48°N.

In [3]:
lo, hi = (pd.Timestamp(s).value // 10**9 for s in (START, END))
# scripts/prepare_zenodo.py already renamed the archive's coded columns, set the
# "not available" codes to NaN, dropped duplicates and filtered the noise
ais = pd.read_parquet(AIS_PARQUET, filters=[("unix_time", ">=", lo), ("unix_time", "<", hi)])
n_raw = len(ais)
ais = (ais.drop_duplicates(["mmsi", "unix_time"])
          .sort_values(["mmsi", "unix_time"]).reset_index(drop=True))
ais["speed_over_ground"] = ais.speed_over_ground.where(ais.speed_over_ground < 102.3)
ais["true_heading"] = ais.true_heading.where(ais.true_heading < 360)
ais = gpd.GeoDataFrame(ais, geometry=gpd.points_from_xy(ais.lon, ais.lat), crs=4326).to_crs(CRS)
ais["x"], ais["y"] = ais.geometry.x, ais.geometry.y
print(f"{START} .. {END}: {n_raw:,} AIS messages -> {len(ais):,} after removing duplicates, "
      f"{ais.mmsi.nunique():,} vessels")

2016-02-01 .. 2016-02-04: 391,836 AIS messages -> 388,164 after removing duplicates, 347 vessels


---
## 2 · Annotation: sea state at every position

The ocean file is a 301 × 181 grid every 3 hours; 43,206 of its cells are sea,
the rest land (undefined values). Three of its variables are used here:

| named here | meaning | unit |
|---|---|---|
| `bottom_depth` (`dpt` in the source) | bottom depth | m |
| `wave_height` (`hs`) | significant wave height | m |
| `wave_direction` (`dir`) | mean wave direction, "from", a **bearing** | degrees |

Annotation interpolates each variable across the grid to the vessel's position
(inverse-distance weighting over the 4 nearest cells), then through time to its
timestamp (DLT, linear between the 3-hourly epochs). Wave direction is a
bearing, so it is interpolated on the unit circle; 350° and 10° average to 0°,
not 180°. Depth does not change in time, so it needs the spatial stage only.

In [4]:
# land cells, "no value" codes and noise were dealt with by scripts/prepare_zenodo.py
oc = pd.read_parquet(OCEAN_PARQUET, filters=[("unix_time", ">=", lo - 3 * 3600),    # bracket the window
                                             ("unix_time", "<=", hi + 3 * 3600)])
gx, gy = Transformer.from_crs(4326, CRS, always_xy=True).transform(oc.lon.values, oc.lat.values)
oc["x"], oc["y"] = gx, gy
print(f"{oc[['lon', 'lat']].drop_duplicates().shape[0]:,} sea points x {oc.unix_time.nunique()} epochs "
      f"({pd.to_datetime(oc.unix_time.min(), unit='s')} .. {pd.to_datetime(oc.unix_time.max(), unit='s')})")


def stations(var):
    return pd.DataFrame({"x": oc.x, "y": oc.y, "t": oc.unix_time.astype(float), "value": oc[var]})


x, y, t = ais.x.to_numpy(), ais.y.to_numpy(), ais.unix_time.to_numpy(float)

t0 = time.time()
grid0 = oc[oc.unix_time == oc.unix_time.min()]                      # depth: one epoch, spatial stage only
ais["bottom_depth"] = IDW(k=4).fit(grid0.x, grid0.y, grid0.bottom_depth)(x, y)
for var, circular in (("wave_height", False), ("wave_direction", True)):
    ais[var] = annotate_points(stations(var), x, y, t,
                               spatial_method=IDW(k=4, circular=circular),
                               temporal_method=DLT(circular=circular), circular=circular)
    print(f"annotated {var} ({time.time() - t0:.0f}s)")
ais[["bottom_depth", "wave_height", "wave_direction"]].describe().round(2)

43,206 sea points x 26 epochs (2016-01-31 23:00:00 .. 2016-02-04 02:00:00)


annotated wave_height (4s)


annotated wave_direction (10s)


,bottom_depth,wave_height,wave_direction
count,388164.00,388164.00,388164.00
mean,33.89,2.09,254.54
std,105.30,1.07,17.31
min,2.59,0.24,192.44
25%,23.66,1.42,241.36
50%,24.22,1.81,246.92
75%,25.47,2.20,269.69
max,4313.68,6.33,330.23


---
## 3 · Compress

`compress` takes the spatial threshold `epsilon` (metres) and one threshold per
context variable, **in that variable's own units**. Each variable is tested on
its own and a point is kept if **any** of them is exceeded. Here: 1 km, wave
height 0.15 m, wave direction 1.5°, bottom depth 80 m, with each vessel as one
trajectory (`by="mmsi"`).

The result is a row subset of the input, so every column of every retained
point survives. With no thresholds it is plain Douglas–Peucker.

Two modes:

* **`post`** (default) is the method as published: errors are measured once,
  against the Douglas–Peucker result, and every point over its threshold is
  added.
* **`iterative`** adds each segment's worst point, measures again, and repeats,
  so every dropped point ends up within its thresholds of the final trajectory.

**Which mode?** `post` is the method as published; `iterative` usually keeps
the context error of every dropped point within its threshold, with fewer
points.

**Unfiltered, as in the paper.** The calls below pass `filter_window=None`: the paper's experiment
compresses the annotated positions as they are. `compress` otherwise median-filters the positions and
every threshold column first, over a 30 s window, so the cell after the table repeats the three runs
with that default.


In [5]:
EPS = 1_000.0
THRESHOLDS = {"wave_height": 0.15, "wave_direction": 1.5, "bottom_depth": 80.0}
CIRCULAR = ["wave_direction"]
N = len(ais)

runs = {"DP": dict(thresholds=None)}
runs |= {f"CADP {m}": dict(thresholds=THRESHOLDS, circular=CIRCULAR, mode=m)
         for m in ("post", "iterative")}

results, rows = {}, []
for label, kw in runs.items():
    t0 = time.perf_counter()
    results[label] = compress(ais, EPS, time_column="unix_time", by="mmsi",
                              filter_window=None, **kw)
    rows.append({"method": label, "kept": len(results[label]),
                 "R (%)": round(compression_ratio(len(results[label]), N), 2),
                 "seconds": round(time.perf_counter() - t0, 2)})
pd.DataFrame(rows).set_index("method")

,kept,R (%),seconds
method,,,
DP,2360,99.39,0.26
CADP post,182406,53.01,0.35
CADP iterative,6433,98.34,0.61


**With the default filter.** The same three runs without `filter_window=None`: the positions, wave
height, wave direction (on the circle) and depth are median-filtered over 30 s before selection, and the
kept rows carry the filtered values. On this data the default keeps 2,298 points for Douglas–Peucker,
182,837 for CADP `post` and 6,351 for `iterative`, against 2,360, 182,406 and 6,433 unfiltered.

In [6]:
rows_default = []
for label, kw in runs.items():
    t0 = time.perf_counter()
    res = compress(ais, EPS, time_column="unix_time", by="mmsi", **kw)
    rows_default.append({"method": label, "kept": len(res),
                         "R (%)": round(compression_ratio(len(res), N), 2),
                         "seconds": round(time.perf_counter() - t0, 2)})
pd.DataFrame(rows_default).set_index("method")

,kept,R (%),seconds
method,,,
DP,2298,99.41,0.77
CADP post,182837,52.90,1.19
CADP iterative,6351,98.36,1.44


**Measuring a result.** Three questions: how much was removed (the
compression ratio), how close the shape stayed (DTW similarity, lower is
closer), and how faithfully the context survived: for every dropped point, how
far its context lies from the interpolation between its *final* neighbours,
the quantity CADP's thresholds are about.

DTW is quadratic in trajectory length, so it is computed on a fixed sample of
vessels, the same sample for every method.

In [7]:
groups = {k: g for k, g in ais.groupby("mmsi", sort=False)}
rng = np.random.default_rng(0)
SAMPLE = sorted(rng.choice([k for k, g in groups.items() if 100 <= len(g) <= 5000],
                           size=min(50, sum(100 <= len(g) <= 5000 for g in groups.values())),
                           replace=False))


def dtw_km(result):
    kept = set(result.index)
    total = 0.0
    for k in SAMPLE:
        g = groups[k]
        xy = np.column_stack([g.x, g.y])
        total += dtw(xy, xy[g.index.isin(kept)])
    return total / 1000


def context_fidelity(result, var):
    kept, errs = set(result.index), []
    for g in groups.values():
        m = g.index.isin(kept)
        e = context_errors(g.unix_time.to_numpy(float), g[var].to_numpy(float), m,
                           circular=var in CIRCULAR)
        errs.append(e[~m])
    e = np.concatenate(errs)
    e = e[np.isfinite(e)]
    return np.sqrt(np.mean(e**2)), e.max()


table = []
for label, res in results.items():
    row = {"method": label, "kept": len(res), "DTW (km)": round(dtw_km(res))}
    for var in THRESHOLDS:
        rmse, worst = context_fidelity(res, var)
        row[f"{var} RMSE"], row[f"{var} max"] = round(rmse, 3), round(worst, 2)
    table.append(row)
print(f"DTW over {len(SAMPLE)} sample vessels; context errors over every dropped point")
pd.DataFrame(table).set_index("method")

DTW over 50 sample vessels; context errors over every dropped point


,kept,DTW (km),wave_height RMSE,wave_height max,wave_direction RMSE,wave_direction max,bottom_depth RMSE,bottom_depth max
method,,,,,,,,
DP,2360,196322,0.242,3.41,5.604,73.96,38.594,3433.76
CADP post,182406,40109,0.048,0.38,0.668,9.14,3.145,110.44
CADP iterative,6433,82039,0.041,0.15,0.470,1.50,2.300,78.37


Thresholds in the table above: `wave_height` 0.15 m, `wave_direction` 1.5°, `bottom_depth` 80 m. In
`iterative` mode no dropped point exceeds them (the `max` columns). In `post`
mode they are tested against the Douglas–Peucker neighbours, and once points
are added those are no longer the neighbours, so the final error is not bounded.

---
## 4 · One vessel up close

The busiest vessel that actually travels, shown in three panels: its track over
the bottom depth with the points each method keeps, and below it the wave
height it met. The plots are interactive: zoom with the mouse wheel or the box
tool, hover a point to read it, click a legend entry to mute it. The three map
panels are linked, so zooming one zooms all.

In [8]:
def path_km(g):
    return np.hypot(np.diff(g.x), np.diff(g.y)).sum() / 1000

moving = sorted(groups, key=lambda k: -len(groups[k]) * (path_km(groups[k]) > 50))
vid = moving[0]
g = groups[vid]
sel = {m: cadp_select(g.x, g.y, g.unix_time.to_numpy(float), EPS, mode=m,
                      contexts=[Context(g[v].to_numpy(float), thr, v in CIRCULAR, v)
                                for v, thr in THRESHOLDS.items()])
       for m in ("post", "iterative")}
print(f"vessel {vid}: {len(g):,} points, {path_km(g):,.0f} km travelled\n")
for m, s in sel.items():
    print(s.summary(), "\n")

vessel 228762000: 30,850 points, 173 km travelled

30850 points -> 12630 kept (59.06% compression; post mode)
  Douglas-Peucker      22
  + context wave_height  9703
  + context wave_direction 11266
  + context bottom_depth 0 

30850 points -> 66 kept (99.79% compression; iterative mode, 4 rounds)
  Douglas-Peucker      22
  + context wave_height  2
  + context wave_direction 42
  + context bottom_depth 0 



In [9]:
to_merc = Transformer.from_crs(CRS, 3857, always_xy=True)   # web mercator, for the basemap
v = pd.DataFrame({"time": pd.to_datetime(g.unix_time.to_numpy(), unit="s"),
                  "speed_over_ground": g.speed_over_ground.to_numpy(), "bottom_depth": g.bottom_depth.to_numpy(),
                  "wave_height": g.wave_height.to_numpy(), "wave_direction": g.wave_direction.to_numpy()})
v["mx"], v["my"] = to_merc.transform(g.x.to_numpy(), g.y.to_numpy())

pad = 15_000
cells = grid0[grid0.x.between(g.x.min() - pad, g.x.max() + pad)
              & grid0.y.between(g.y.min() - pad, g.y.max() + pad)].copy()
cells["mx"], cells["my"] = to_merc.transform(cells.x.to_numpy(), cells.y.to_numpy())
depth = cells.hvplot.points("mx", "my", c="bottom_depth", cmap="Blues", clim=(0, 150), marker="square",
                            size=60, alpha=0.55, colorbar=True, clabel="bottom depth (m)",
                            hover_cols=["bottom_depth"])
hover = ["time", "speed_over_ground", "bottom_depth", "wave_height", "wave_direction"]


def panel(title, mask, added=None):
    layers = [hv.element.tiles.OSM().opts(alpha=0.6), depth,
              v.hvplot.paths("mx", "my", color="#999999", line_width=0.7, label="raw track"),
              v[mask].hvplot.paths("mx", "my", color="#cf222e", line_width=1.6,
                                   label="compressed"),
              v[sel["post"].dp].hvplot.points("mx", "my", color=INK, size=40,
                                              hover_cols=hover, label="kept by DP")]
    if added is not None:
        layers.append(v[added].hvplot.points("mx", "my", color=ORANGE, size=22,
                                              hover_cols=hover, label="added for context"))
    return hv.Overlay(layers).opts(title=f"{title}: {mask.sum():,} of {len(v):,} points",
                                   width=420, height=440, xaxis=None, yaxis=None,
                                   legend_position="bottom_right")


hv.Layout([panel("Douglas–Peucker", sel["post"].dp),
           panel("CADP · post", sel["post"].mask, sel["post"].added),
           panel("CADP · iterative", sel["iterative"].mask, sel["iterative"].added)]).cols(3)

:Layout
   .Overlay.I   :Overlay
      .Tiles.I           :Tiles   [x,y]
      .Points.I          :Points   [mx,my]   (bottom_depth)
      .Path.Raw_track    :Path   [mx,my]
      .Path.Compressed   :Path   [mx,my]
      .Points.Kept_by_DP :Points   [mx,my]   (time,speed_over_ground,bottom_depth,wave_height,wave_direction)
   .Overlay.II  :Overlay
      .Tiles.I                  :Tiles   [x,y]
      .Points.I                 :Points   [mx,my]   (bottom_depth)
      .Path.Raw_track           :Path   [mx,my]
      .Path.Compressed          :Path   [mx,my]
      .Points.Kept_by_DP        :Points   [mx,my]   (time,speed_over_ground,bottom_depth,wave_height,wave_direction)
      .Points.Added_for_context :Points   [mx,my]   (time,speed_over_ground,bottom_depth,wave_height,wave_direction)
   .Overlay.III :Overlay
      .Tiles.I                  :Tiles   [x,y]
      .Points.I                 :Points   [mx,my]   (bottom_depth)
      .Path.Raw_track           :Path   [mx,my]
      .Path.Compressed          :Path   [mx,my]
      .Points.Kept_by_DP        :Points   [mx,my]   (time,speed_over_ground,bottom_depth,wave_height,wave_direction)
      .Points.Added_for_context :Points   [mx,my]   (time,speed_over_ground,bottom_depth,wave_height,wave_direction)

In [10]:
series = [v.hvplot.line("time", "wave_height", color="#999999", line_width=1, label="annotated wave height")]
for m, colour, size in (("post", ORANGE, 12), ("iterative", GREEN, 40)):
    k = sel[m].mask
    label = f"{m}: {k.sum():,} points"
    series += [v[k].hvplot.line("time", "wave_height", color=colour, line_width=1.2, label=label),
               v[k].hvplot.scatter("time", "wave_height", color=colour, size=size, label=label,
                                   hover_cols=["speed_over_ground", "bottom_depth", "wave_direction"])]
hv.Overlay(series).opts(
    width=1000, height=330, ylabel="significant wave height (m)", legend_position="top_left",
    title=f"vessel {vid}: what each compressed trajectory says about the sea it crossed")

:Overlay
   .Curve.Annotated_wave_height            :Curve   [time]   (wave_height)
   .Curve.Post_colon_12_comma_630_points   :Curve   [time]   (wave_height)
   .Scatter.Post_colon_12_comma_630_points :Scatter   [time]   (wave_height,speed_over_ground,bottom_depth,wave_direction)
   .Curve.Iterative_colon_66_points        :Curve   [time]   (wave_height)
   .Scatter.Iterative_colon_66_points      :Scatter   [time]   (wave_height,speed_over_ground,bottom_depth,wave_direction)

---
## 5 · Choosing thresholds

A threshold is the largest context error you accept for a dropped point, so it
is best chosen in the variable's own terms: how much wave height or depth
difference matters for the analysis that follows. The sweep below shows the
cost of each choice, one variable at a time, with ε fixed at 1 km: a larger
threshold compresses more and keeps the shape less closely.

In [11]:
SWEEPS = {"wave_direction": [1.0, 1.5, 2.0, 2.5], "wave_height": [0.05, 0.1, 0.15, 0.25, 0.5],
          "bottom_depth": [10.0, 20.0, 40.0, 80.0, 160.0]}
sweep = []
for var, thresholds in SWEEPS.items():
    for thr in thresholds:
        for mode in ("post", "iterative"):
            res = compress(ais, EPS, thresholds={var: thr}, circular=CIRCULAR if var in CIRCULAR else (),
                           time_column="unix_time", by="mmsi", mode=mode, filter_window=None)
            rmse, worst = context_fidelity(res, var)
            sweep.append({"variable": var, "threshold": thr, "mode": mode,
                          "R (%)": compression_ratio(len(res), N), "DTW (km)": dtw_km(res),
                          "ctx RMSE": rmse, "ctx max": worst})
sweep = pd.DataFrame(sweep)
sweep[sweep.variable == "wave_direction"].round(3)

,variable,threshold,mode,R (%),DTW (km),ctx RMSE,ctx max
0,wave_direction,1.0,post,50.641,49423.371,0.607,8.811
1,wave_direction,1.0,iterative,97.929,81287.769,0.343,0.999
2,wave_direction,1.5,post,61.031,60830.169,0.749,9.136
3,wave_direction,1.5,iterative,98.381,94640.975,0.473,1.500
4,wave_direction,2.0,post,67.733,69391.073,0.964,9.136
5,wave_direction,2.0,iterative,98.633,107305.344,0.614,2.000
6,wave_direction,2.5,post,72.230,79326.832,1.278,9.136
7,wave_direction,2.5,iterative,98.793,113298.219,0.760,2.499


In [12]:
names = {"wave_direction": "wave direction (°)", "wave_height": "wave height (m)", "bottom_depth": "bottom depth (m)"}
plots = []
for metric in ("R (%)", "DTW (km)", "ctx RMSE"):
    for var in SWEEPS:
        layers = []
        for mode, colour in (("post", ORANGE), ("iterative", GREEN)):
            d = sweep[(sweep.variable == var) & (sweep["mode"] == mode)]
            layers += [d.hvplot.line("threshold", metric, color=colour, label=mode),
                       d.hvplot.scatter("threshold", metric, color=colour, size=50, label=mode,
                                        hover_cols=["R (%)", "DTW (km)", "ctx RMSE", "ctx max"])]
        plots.append(hv.Overlay(layers).opts(
            width=340, height=240, xlabel=f"threshold, {names[var]}", ylabel=metric,
            yformatter="%.4g", title=names[var] if metric == "R (%)" else "",
            show_legend=metric == "R (%)", legend_position="bottom_right"))
hv.Layout(plots).cols(3).opts(shared_axes=False)

:Layout
   .Overlay.I    :Overlay
      .Curve.Post        :Curve   [threshold]   (R (%))
      .Scatter.Post      :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (R (%))
      .Scatter.Iterative :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
   .Overlay.II   :Overlay
      .Curve.Post        :Curve   [threshold]   (R (%))
      .Scatter.Post      :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (R (%))
      .Scatter.Iterative :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
   .Overlay.III  :Overlay
      .Curve.Post        :Curve   [threshold]   (R (%))
      .Scatter.Post      :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (R (%))
      .Scatter.Iterative :Scatter   [threshold]   (R (%),DTW (km),ctx RMSE,ctx max)
   .Overlay.IV   :Overlay
      .Curve.Post        :Curve   [threshold]   (DTW (km))
      .Scatter.Post      :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (DTW (km))
      .Scatter.Iterative :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
   .Overlay.V    :Overlay
      .Curve.Post        :Curve   [threshold]   (DTW (km))
      .Scatter.Post      :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (DTW (km))
      .Scatter.Iterative :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
   .Overlay.VI   :Overlay
      .Curve.Post        :Curve   [threshold]   (DTW (km))
      .Scatter.Post      :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
      .Curve.Iterative   :Curve   [threshold]   (DTW (km))
      .Scatter.Iterative :Scatter   [threshold]   (DTW (km),R (%),ctx RMSE,ctx max)
   .Overlay.VII  :Overlay
      .Curve.Post        :Curve   [threshold]   (ctx RMSE)
      .Scatter.Post      :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)
      .Curve.Iterative   :Curve   [threshold]   (ctx RMSE)
      .Scatter.Iterative :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)
   .Overlay.VIII :Overlay
      .Curve.Post        :Curve   [threshold]   (ctx RMSE)
      .Scatter.Post      :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)
      .Curve.Iterative   :Curve   [threshold]   (ctx RMSE)
      .Scatter.Iterative :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)
   .Overlay.IX   :Overlay
      .Curve.Post        :Curve   [threshold]   (ctx RMSE)
      .Scatter.Post      :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)
      .Curve.Iterative   :Curve   [threshold]   (ctx RMSE)
      .Scatter.Iterative :Scatter   [threshold]   (ctx RMSE,R (%),DTW (km),ctx max)

---
## 6 · With MovingPandas

`compress` accepts MovingPandas objects directly and returns the same type,
with every column of the retained points, the annotated context included. A
`TrajectoryCollection` is compressed trajectory by trajectory.

In [13]:
import movingpandas as mpd

frame = ais.assign(time=pd.to_datetime(ais.unix_time, unit="s"))
busiest = frame.mmsi.value_counts().index[:20]
collection = mpd.TrajectoryCollection(frame[frame.mmsi.isin(busiest)],
                                      traj_id_col="mmsi", t="time")
small = compress(collection, EPS, thresholds=THRESHOLDS, circular=CIRCULAR, mode="iterative")

before = sum(len(tr.df) for tr in collection.trajectories)
after = sum(len(tr.df) for tr in small.trajectories)
print(f"{len(collection.trajectories)} trajectories: {before:,} -> {after:,} points "
      f"(R = {compression_ratio(after, before):.2f}%)")

traj = small.trajectories[0]
print(type(traj).__name__, traj.id, f"{len(traj.df):,} points")
traj.df[["speed_over_ground", "bottom_depth", "wave_height", "wave_direction"]].head()

20 trajectories: 255,769 -> 2,072 points (R = 99.19%)
Trajectory 205309000 56 points


,speed_over_ground,bottom_depth,wave_height,wave_direction
time,,,,
2016-02-01 00:01:54,0.0,24.187998,1.916726,241.319915
2016-02-02 04:58:53,0.0,24.187989,2.283247,240.953221
2016-02-02 11:01:53,0.0,24.188002,1.866555,242.502479
2016-02-03 04:58:53,0.0,24.187938,1.588238,248.102670
2016-02-03 07:58:12,4.2,24.257650,1.575985,251.605939


---
## 7 · Speed

Douglas–Peucker and both CADP modes on growing slices of the longest trajectory
in the window, ε = 10 m, the mean of 10 runs. Absolute times depend on the
machine; the point is the overhead of the context test over plain DP.

In [14]:
longest = max(groups.values(), key=len)
sizes = [n for n in (1_000, 5_000, 10_000, 50_000, 100_000) if n < len(longest)] + [len(longest)]
timing = []
for n in sizes:
    h = longest.iloc[:n]
    hx, hy, ht = h.x.to_numpy(), h.y.to_numpy(), h.unix_time.to_numpy(float)
    row = {"n": n}
    jobs = {"DP": lambda: douglas_peucker(hx, hy, 10.0)}
    for var, thr in THRESHOLDS.items():
        for mode in ("post", "iterative"):
            jobs[f"{mode} {var}"] = (lambda v=var, th=thr, m=mode: cadp_select(
                hx, hy, ht, 10.0, contexts=[Context(h[v].to_numpy(float), th, v in CIRCULAR)],
                mode=m))
    for name, job in jobs.items():
        times = []
        for _ in range(10):
            t0 = time.perf_counter(); job(); times.append((time.perf_counter() - t0) * 1000)
        row[name] = round(float(np.mean(times)), 1)
    timing.append(row)
print(f"longest trajectory in the window: {len(longest):,} points; times in ms (mean of 10)")
pd.DataFrame(timing).set_index("n")

longest trajectory in the window: 30,850 points; times in ms (mean of 10)


,DP,post wave_height,iterative wave_height,post wave_direction,iterative wave_direction,post bottom_depth,iterative bottom_depth
n,,,,,,,
1000,0.0,0.1,0.1,0.1,0.1,0.1,0.1
5000,0.1,0.2,0.2,0.4,0.3,0.2,0.2
10000,0.2,0.3,0.4,0.5,0.5,0.4,0.4
30850,13.0,13.3,15.6,14.1,17.3,13.4,13.4


---
## Notes

* **Change the window** with `START` and `END` in the settings cell. Annotation
  takes a few seconds per variable per day of data (14 s for the two variables over
  the three days here). The ocean
  file covers February 2016; other months are in the same archive
  (`oc_october.csv` … `oc_march.csv`).
* **Your own data**: any trajectories in a projected CRS, with context as
  ordinary columns, work the same way. Annotate them with `catraj.annotation`,
  or use columns recorded with the positions, like speed.
* **The modes**: `help(catraj.compression.cadp)` explains both, and why they
  differ most where context changes smoothly along a long straight leg.

**Citing.** Please cite:

* **MovingPandas**, which `catraj` builds on: Graser, A. (2019). *MovingPandas: Efficient Structures for Movement Data in Python.* GI_Forum 2019, 7(1), 54–68. [doi:10.1553/giscience2019_01_s54](https://doi.org/10.1553/giscience2019_01_s54)
* **CADP**, the method: Mehri, S., Hooshangi, N., Mahdizadeh Gharakhanlou, N. (2025). ISPRS Int. J. Geo-Inf. 14(2), 58. [doi:10.3390/ijgi14020058](https://doi.org/10.3390/ijgi14020058)
* **The data**: Ray, C. et al. (2019). Data in Brief 25, 104141. [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141); Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595)
* **The sea state**: Boudière, E. et al. (2013). International Journal of Marine Energy 3–4, e40–e52. [doi:10.1016/j.ijome.2013.11.010](https://doi.org/10.1016/j.ijome.2013.11.010)

Anything derived from the AIS data, including the outputs of this notebook,
falls under CC BY-NC-SA 4.0.